# Evaluacion de Tres Sistemas (Analisis de Tests Pre-ejecutados)

Notebook para comparar tres funciones de perdida usando resultados ya generados en `src/benchmark_results`:
- Sistema A: `mse`
- Sistema B: `match_count`
- Sistema C: `sigmoid`

Este notebook **no ejecuta tests**. Solo carga resultados existentes y realiza analisis estadistico e inferencia de hipotesis.

## Marco metodologico y preguntas de investigacion

Este notebook implementa un experimento comparativo **pareado de 3 sistemas** sobre el mismo conjunto de casos, usando salidas precomputadas.

Preguntas de investigacion:
1. Existen diferencias globales en `root_match_rate` entre `mse`, `match_count` y `sigmoid`?
2. Existen diferencias globales en cobertura (`coverage_combined`) entre los 3 sistemas?
3. Existen diferencias globales en tiempo (`time_seconds`) entre los 3 sistemas?
4. Existen diferencias en robustez operativa (`no_sr_results`, `branch_count_correct`, casos perfectos)?

Logica de lectura:
1. Primero se cargan y validan los 3 tests (`test_mse`, `test_match_count`, `test_sigmoid`).
2. Luego se construyen tablas pareadas por caso para comparacion justa.
3. Finalmente se ejecutan pruebas globales y post-hoc, con conclusiones automaticas.

## 1. Configurar entorno y dependencias

Esta celda detecta automáticamente la raíz del proyecto y valida que el kernel use un entorno virtual activo.

In [1]:
import gc
import os
import sys
import json
import time
import random
from itertools import combinations
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wilcoxon, binomtest, friedmanchisquare, chi2


def find_project_root(start_path: Path) -> Path:
    """Encuentra la raíz del proyecto buscando src/config.py hacia arriba."""
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'src' / 'config.py').exists() and (candidate / 'src' / 'benchmark').exists():
            return candidate
    raise FileNotFoundError(
        'No se encontró la raíz del proyecto. Ejecuta este notebook desde dentro del repo.'
    )


CURRENT_PY = Path(sys.executable).resolve()
IN_VENV = (hasattr(sys, 'base_prefix') and sys.prefix != sys.base_prefix) or bool(os.environ.get('VIRTUAL_ENV'))

print('Python actual  :', CURRENT_PY)
if IN_VENV:
    print('Entorno virtual:', os.environ.get('VIRTUAL_ENV', '(detectado por sys.prefix)'))
else:
    print('Aviso: no se detectó entorno virtual activo. Se recomienda usar un venv del proyecto.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / 'src'
BENCH_DIR = SRC_DIR / 'benchmark'
RESULTS_ROOT = SRC_DIR / 'benchmark_results'
print('Project root   :', PROJECT_ROOT)

# Paths de módulos
for p in [
    SRC_DIR,
    BENCH_DIR,
    SRC_DIR / '1_equation_definition',
    SRC_DIR / '2_parameter_grid',
    SRC_DIR / '3_zero_finding',
    SRC_DIR / '4_data_preparation',
    SRC_DIR / '5_symbolic_regression',
    SRC_DIR / '6_expression_builder',
]:
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)

import config as cfg
from test_cases import TEST_CASES

print('OK. Dependencias cargadas.')
print('Casos benchmark:', len(TEST_CASES))
print('Config SR desde src/config.py ->')
print(f"  NITERATIONS={cfg.NITERATIONS}, MIN_POINTS={cfg.MIN_POINTS}, MAX_ITERATIONS={cfg.MAX_ITERATIONS}")

Python actual  : /usr/bin/python3.12
Entorno virtual: /home/noel/Disco D/4to_Anno/tesis/Tesis/.venv
Project root   : /home/noel/Disco D/4to_Anno/tesis/Tesis
OK. Dependencias cargadas.
Casos benchmark: 43
Config SR desde src/config.py ->
  NITERATIONS=500, MIN_POINTS=5, MAX_ITERATIONS=None


In [2]:
# Conclusiones automáticas de la etapa 1 (entorno y configuración base)
setup_summary = pd.DataFrame([
    {
        'python_executable': str(CURRENT_PY),
        'in_virtual_env': bool(IN_VENV),
        'project_root_exists': bool(PROJECT_ROOT.exists()),
        'src_exists': bool(SRC_DIR.exists()),
        'benchmark_dir_exists': bool(BENCH_DIR.exists()),
        'benchmark_results_exists': bool(RESULTS_ROOT.exists()),
        'n_test_cases': int(len(TEST_CASES)),
        'niterations': int(cfg.NITERATIONS),
        'min_points': int(cfg.MIN_POINTS),
        'max_iterations': cfg.MAX_ITERATIONS,
    }
])
display(setup_summary)

if IN_VENV and SRC_DIR.exists() and BENCH_DIR.exists() and RESULTS_ROOT.exists():
    print('Conclusion etapa 1: entorno correctamente configurado para análisis reproducible de resultados pre-ejecutados.')
else:
    print('Conclusion etapa 1: revisar entorno/rutas antes de analizar resultados.')

,python_executable,in_virtual_env,project_root_exists,src_exists,benchmark_dir_exists,benchmark_results_exists,n_test_cases,niterations,min_points,max_iterations
0,/usr/bin/python3.12,True,True,True,True,True,43,500,5,None


Conclusion etapa 1: entorno correctamente configurado para análisis reproducible de resultados pre-ejecutados.


## 2. Cargar datos de referencia y resultados de 3 tests

Se carga el catálogo de referencia (`TEST_CASES`) y los resultados ya ejecutados desde:
- `benchmark_results/test_mse`
- `benchmark_results/test_match_count`
- `benchmark_results/test_sigmoid`

In [4]:
reference_df = pd.DataFrame([
    {
        'name': tc['name'],
        'category': tc['category'],
        'difficulty': tc['difficulty'],
        'equation': tc['equation'],
        'roots_expected': len(tc['expected_roots'])
    }
    for tc in TEST_CASES
])

SYSTEM_MAP = {
    'mse': {'folder': RESULTS_ROOT / 'test_mse', 'system': 'A'},
    'match_count': {'folder': RESULTS_ROOT / 'test_match_count', 'system': 'B'},
    'sigmoid': {'folder': RESULTS_ROOT / 'test_sigmoid', 'system': 'C'},
}


def load_test_evaluations(folder: Path, loss_mode: str, system_label: str) -> pd.DataFrame:
    eval_path = folder / 'evaluations.json'
    if not eval_path.exists():
        raise FileNotFoundError(f'Falta archivo requerido para {loss_mode}: {eval_path}')

    with open(eval_path, 'r', encoding='utf-8') as f:
        payload = json.load(f)

    df = pd.DataFrame(payload)
    if df.empty:
        raise ValueError(f'Archivo evaluations.json vacío para {loss_mode}: {eval_path}')

    df['loss_mode'] = loss_mode
    df['system'] = system_label
    if 'repeat_id' not in df.columns:
        df['repeat_id'] = 1

    if 'coverage_combined' not in df.columns:
        df['coverage_combined'] = df.get('average_coverage', np.nan)
    if 'error' not in df.columns:
        df['error'] = df.get('any_error', None)
    if 'branch_count_correct' not in df.columns:
        df['branch_count_correct'] = False

    # Variables binarias para hipótesis adicionales
    df['perfect_case'] = (pd.to_numeric(df.get('root_match_rate', 0.0), errors='coerce').fillna(0.0) >= 1.0).astype(int)
    df['no_sr_case'] = (df.get('status', '').astype(str) == 'no_sr_results').astype(int)

    return df


df_mse_raw = load_test_evaluations(SYSTEM_MAP['mse']['folder'], 'mse', 'A')
df_match_raw = load_test_evaluations(SYSTEM_MAP['match_count']['folder'], 'match_count', 'B')
df_sigmoid_raw = load_test_evaluations(SYSTEM_MAP['sigmoid']['folder'], 'sigmoid', 'C')

run_paths_df = pd.DataFrame([
    {'loss_mode': mode, 'system': cfg_info['system'], 'folder': str(cfg_info['folder'])}
    for mode, cfg_info in SYSTEM_MAP.items()
])

print('Directorios de tests:')
display(run_paths_df)
print('Shapes cargados:')
print('  mse        :', df_mse_raw.shape)
print('  match_count:', df_match_raw.shape)
print('  sigmoid    :', df_sigmoid_raw.shape)

display(reference_df.head())

FileNotFoundError: Falta archivo requerido para mse: /home/noel/Disco D/4to_Anno/tesis/Tesis/src/benchmark_results/test_mse/evaluations.json

In [7]:
# Conclusiones del bloque 2 (catálogo y carga de tests)
load_summary = pd.DataFrame([
    {
        'loss_mode': 'mse',
        'rows': int(len(df_mse_raw)),
        'n_cases': int(df_mse_raw['name'].nunique()) if not df_mse_raw.empty and 'name' in df_mse_raw.columns else 0,
        'loaded_ok': bool(not df_mse_raw.empty),
    },
    {
        'loss_mode': 'match_count',
        'rows': int(len(df_match_raw)),
        'n_cases': int(df_match_raw['name'].nunique()) if not df_match_raw.empty and 'name' in df_match_raw.columns else 0,
        'loaded_ok': bool(not df_match_raw.empty),
    },
    {
        'loss_mode': 'sigmoid',
        'rows': int(len(df_sigmoid_raw)),
        'n_cases': int(df_sigmoid_raw['name'].nunique()) if not df_sigmoid_raw.empty and 'name' in df_sigmoid_raw.columns else 0,
        'loaded_ok': bool(not df_sigmoid_raw.empty),
    },
])
display(load_summary)

dist_categorias = reference_df['category'].value_counts().rename_axis('category').reset_index(name='n_cases')
dist_dificultad = reference_df['difficulty'].value_counts().rename_axis('difficulty').reset_index(name='n_cases')

print('Distribución por categoría del catálogo:')
display(dist_categorias)
print('Distribución por dificultad del catálogo:')
display(dist_dificultad)

if load_summary['loaded_ok'].all():
    print('CONCLUSION BLOQUE 2: los 3 tests fueron cargados correctamente y están listos para comparación.')
else:
    print('CONCLUSION BLOQUE 2: faltan uno o más tests; revisar carpetas en benchmark_results antes del análisis.')

,n_casos_catalogo,n_categorias,n_dificultades,hay_corrida_previa_A,hay_corrida_previa_B
0,43,6,3,False,False


Distribución por categoría:


,category,n_cases
0,quadratic,14
1,linear,10
2,special,8
3,cubic,6
4,quartic,4
5,quintic,1


Distribución por dificultad:


,difficulty,n_cases
0,easy,19
1,medium,19
2,hard,5


CONCLUSION BLOQUE 2: no hay resultados previos cargados; se ejecutará una evaluación A/B nueva desde cero.


## 3. Normalizar formato y validar consistencia entre 3 sistemas

Se validan columnas, tipos, faltantes, duplicados y conjunto común de casos para asegurar comparación pareada justa.

In [ ]:
REQUIRED_COLS = [
    'name', 'category', 'difficulty', 'repeat_id', 'system', 'loss_mode',
    'status', 'time_seconds', 'roots_expected', 'roots_matched',
    'root_match_rate', 'coverage_combined', 'branch_count_correct',
    'perfect_case', 'no_sr_case'
]


def normalize_and_validate(df: pd.DataFrame, label: str = 'df') -> pd.DataFrame:
    if df.empty:
        print(f'{label}: vacío')
        return df.copy()

    out = df.copy()

    missing = [c for c in REQUIRED_COLS if c not in out.columns]
    if missing:
        raise ValueError(f'{label}: faltan columnas {missing}')

    for c in ['name', 'category', 'difficulty', 'system', 'loss_mode', 'status']:
        out[c] = out[c].astype(str)

    for c in [
        'repeat_id', 'time_seconds', 'roots_expected', 'roots_matched',
        'root_match_rate', 'coverage_combined', 'branch_count_correct',
        'perfect_case', 'no_sr_case'
    ]:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    out['branch_count_correct'] = out['branch_count_correct'].fillna(0).astype(int)
    out['perfect_case'] = out['perfect_case'].fillna(0).astype(int)
    out['no_sr_case'] = out['no_sr_case'].fillna(0).astype(int)

    dup = out.duplicated(subset=['name', 'repeat_id', 'loss_mode']).sum()
    print(f'{label}: filas={len(out)}, duplicados(name,repeat_id,loss_mode)={int(dup)}')

    null_report = out[REQUIRED_COLS].isna().sum().reset_index()
    null_report.columns = ['column', 'nulls']
    display(null_report)

    return out


df_mse = normalize_and_validate(df_mse_raw, 'df_mse')
df_match = normalize_and_validate(df_match_raw, 'df_match_count')
df_sigmoid = normalize_and_validate(df_sigmoid_raw, 'df_sigmoid')

keys_mse = set(zip(df_mse['name'], df_mse['repeat_id'])) if not df_mse.empty else set()
keys_match = set(zip(df_match['name'], df_match['repeat_id'])) if not df_match.empty else set()
keys_sigmoid = set(zip(df_sigmoid['name'], df_sigmoid['repeat_id'])) if not df_sigmoid.empty else set()

common_keys = keys_mse & keys_match & keys_sigmoid
print('Casos/repeticiones en común entre los 3 sistemas:', len(common_keys))

if len(common_keys) == 0:
    raise ValueError('No hay casos en común entre los 3 tests. No se puede hacer análisis pareado.')


def keep_common(df: pd.DataFrame) -> pd.DataFrame:
    keys = list(zip(df['name'], df['repeat_id']))
    mask = [k in common_keys for k in keys]
    return df.loc[mask].copy()


df_mse = keep_common(df_mse)
df_match = keep_common(df_match)
df_sigmoid = keep_common(df_sigmoid)

df_all = pd.concat([df_mse, df_match, df_sigmoid], ignore_index=True)
df_all = normalize_and_validate(df_all, 'df_all')

consistency_table = (
    df_all.groupby('loss_mode', as_index=False)
          .agg(
              n_rows=('name', 'count'),
              n_cases=('name', 'nunique'),
              n_repeat=('repeat_id', 'nunique')
          )
)
display(consistency_table)

In [ ]:
# Conclusiones del bloque 3 (consistencia para análisis pareado)
quality_rows = []
for label, df in [('mse', df_mse), ('match_count', df_match), ('sigmoid', df_sigmoid)]:
    dup = int(df.duplicated(subset=['name', 'repeat_id', 'loss_mode']).sum()) if not df.empty else 0
    succ = float((df['status'] == 'success').mean()) if not df.empty else np.nan
    quality_rows.append({
        'dataset': label,
        'is_empty': bool(df.empty),
        'n_rows': int(len(df)),
        'n_cases': int(df['name'].nunique()) if not df.empty else 0,
        'duplicates_name_repeat_loss': dup,
        'success_rate_if_available': succ,
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

if not quality_df['is_empty'].any() and int(quality_df['duplicates_name_repeat_loss'].sum()) == 0:
    print('CONCLUSION BLOQUE 3: datos consistentes para comparación pareada de 3 sistemas.')
else:
    print('CONCLUSION BLOQUE 3: hay vacíos o duplicados; revisar antes de interpretar hipótesis.')

,dataset,is_empty,n_rows,n_cases,duplicates_name_repeat_system,success_rate_if_available
0,A_previo,True,0,0,0,NaN
1,B_previo,True,0,0,0,NaN


CONCLUSION BLOQUE 3: no hay datasets previos para validar; la comparación principal será con la nueva corrida A/B.


## 4. Definir hipótesis y utilidades estadísticas (3 sistemas)

En este bloque se definen:
- El catálogo de hipótesis para 3 sistemas.
- Funciones de agregación y tablas pareadas.
- Utilidades para corrección de múltiples pruebas y tests binarios.

In [ ]:
SYSTEM_ORDER = ['mse', 'match_count', 'sigmoid']
SYSTEM_LABEL = {'mse': 'A', 'match_count': 'B', 'sigmoid': 'C'}
SYSTEM_TITLE = {'mse': 'MSE', 'match_count': 'Match Count', 'sigmoid': 'Sigmoidal'}
PAIRS = list(combinations(SYSTEM_ORDER, 2))
ALPHA = 0.05

HYPOTHESES = pd.DataFrame([
    {
        'id': 'H1',
        'type': 'continua',
        'metric': 'root_match_rate',
        'null_hypothesis': 'La distribución de root_match_rate es igual en mse, match_count y sigmoid.'
    },
    {
        'id': 'H2',
        'type': 'continua',
        'metric': 'coverage_combined',
        'null_hypothesis': 'La distribución de coverage_combined es igual en mse, match_count y sigmoid.'
    },
    {
        'id': 'H3',
        'type': 'continua',
        'metric': 'time_seconds',
        'null_hypothesis': 'La distribución de time_seconds es igual en mse, match_count y sigmoid.'
    },
    {
        'id': 'H4',
        'type': 'binaria',
        'metric': 'perfect_case',
        'null_hypothesis': 'La proporción de casos perfectos es igual en los 3 sistemas.'
    },
    {
        'id': 'H5',
        'type': 'binaria',
        'metric': 'no_sr_case',
        'null_hypothesis': 'La proporción de no_sr_results es igual en los 3 sistemas.'
    },
    {
        'id': 'H6',
        'type': 'binaria',
        'metric': 'branch_count_correct',
        'null_hypothesis': 'La proporción de branch_count_correct es igual en los 3 sistemas.'
    },
])


def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby(['system', 'loss_mode'], as_index=False)
          .agg(
              n_runs=('name', 'count'),
              n_cases=('name', 'nunique'),
              success_rate=('status', lambda s: float((s == 'success').mean())),
              mean_root_match_rate=('root_match_rate', 'mean'),
              mean_coverage=('coverage_combined', 'mean'),
              mean_time_seconds=('time_seconds', 'mean'),
              perfect_rate=('perfect_case', 'mean'),
              no_sr_rate=('no_sr_case', 'mean'),
              branch_count_correct_rate=('branch_count_correct', 'mean'),
          )
    )


def to_wide(metric: str, dropna: bool = True) -> pd.DataFrame:
    wide = (
        df_all
        .pivot_table(index=['name', 'repeat_id'], columns='loss_mode', values=metric, aggfunc='mean')
        .copy()
    )
    for mode in SYSTEM_ORDER:
        if mode not in wide.columns:
            wide[mode] = np.nan
    wide = wide[SYSTEM_ORDER]
    return wide.dropna() if dropna else wide


def holm_correction(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    adjusted = np.full_like(pvalues, np.nan, dtype=float)
    finite_idx = np.where(np.isfinite(pvalues))[0]
    if len(finite_idx) == 0:
        return adjusted

    order = finite_idx[np.argsort(pvalues[finite_idx])]
    m = len(order)
    running_max = 0.0
    for i, idx in enumerate(order):
        adj = (m - i) * pvalues[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


def cochran_q_test(binary_matrix: np.ndarray):
    """Cochran Q para k tratamientos pareados binarios."""
    x = np.asarray(binary_matrix, dtype=float)
    if x.ndim != 2:
        return np.nan, np.nan

    n, k = x.shape
    if n < 3 or k < 3:
        return np.nan, np.nan

    row_sums = np.sum(x, axis=1)
    col_sums = np.sum(x, axis=0)
    total = np.sum(col_sums)

    denominator = (k * total) - np.sum(row_sums ** 2)
    if denominator <= 0:
        return np.nan, np.nan

    numerator = (k - 1) * ((k * np.sum(col_sums ** 2)) - (total ** 2))
    q_stat = numerator / denominator
    p_value = 1.0 - chi2.cdf(q_stat, df=k - 1)
    return float(q_stat), float(p_value)


def pairwise_mcnemar_exact(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x, dtype=int)
    y = np.asarray(y, dtype=int)
    n01 = int(np.sum((x == 0) & (y == 1)))
    n10 = int(np.sum((x == 1) & (y == 0)))
    nd = n01 + n10
    if nd == 0:
        return 1.0, n01, n10, nd
    p_value = float(binomtest(min(n01, n10), n=nd, p=0.5).pvalue)
    return p_value, n01, n10, nd


display(HYPOTHESES)

## 5. Resumen descriptivo de los 3 tests cargados

En esta sección no se ejecuta benchmark. Solo se resume el desempeño observado en los resultados cargados.

In [ ]:
summary_all = aggregate_metrics(df_all)
summary_all = summary_all.sort_values('loss_mode').reset_index(drop=True)

difficulty_summary = (
    df_all.groupby(['loss_mode', 'difficulty'], as_index=False)
          .agg(
              n_cases=('name', 'count'),
              mean_root_match_rate=('root_match_rate', 'mean'),
              mean_coverage=('coverage_combined', 'mean'),
              mean_time_seconds=('time_seconds', 'mean'),
          )
)

status_summary = (
    df_all.groupby(['loss_mode', 'status'], as_index=False)
          .size()
          .rename(columns={'size': 'n'})
)

print('Resumen global por sistema:')
display(summary_all)
print('Resumen por dificultad:')
display(difficulty_summary)
print('Distribución de status por sistema:')
display(status_summary)

Compiling Julia backend...


Total casos a ejecutar: 43
Directorio de salida: /home/noel/Disco D/4to_Anno/tesis/Tesis/src/benchmark_results/ab_notebook_full_20260404_233702

=== Sistema A (mse) | Repetición 1/1 ===


[ Info: Started!



Expressions evaluated per second: 2.500e+05
Progress: 563 / 15000 total iterations (3.753%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           8.673e+00  0.000e+00  y = 2
2           4.000e+00  7.740e-01  y = neg(a)
3           1.623e-14  3.314e+01  y = 2 - a
5           1.185e-14  1.573e-01  y = (0.44859 - a) + 1.5514
7           6.559e-15  2.958e-01  y = (1 - (a + -0.5307)) + 0.4693
16          6.026e-15  9.415e-03  y = ((0.80914 + safe_root9(a)) - (a - 0.19086)) + (safe_ro...
                                      ot9(neg(a)) - -1)
19          4.889e-15  6.969e-02  y = ((safe_root9(a / -0.84433) + 1.8091) + safe_root9(a / ...
                                      0.84433)) - (a - 0.19086)
──────────────────────────────────────────────────────────────────────────────────────────────

[ Info: Final population:
[ Info: Results saved to:


[A] 01/43 completado: linear_01_basic | status=success | match=1.000 | cov=1.000 | t=183.4s
  - /tmp/tmps_rcy31l/20260404_233702_o65b9Y/hall_of_fame.csv


[ Info: Started!


In [ ]:
# Conclusiones del bloque 5 (descriptivo global de 3 sistemas)
mean_match_by_system = summary_all.set_index('loss_mode')['mean_root_match_rate']
mean_cov_by_system = summary_all.set_index('loss_mode')['mean_coverage']
mean_time_by_system = summary_all.set_index('loss_mode')['mean_time_seconds']

best_match_system = mean_match_by_system.idxmax() if len(mean_match_by_system) else 'NA'
best_cov_system = mean_cov_by_system.idxmax() if len(mean_cov_by_system) else 'NA'
fastest_system = mean_time_by_system.idxmin() if len(mean_time_by_system) else 'NA'

print('CONCLUSION BLOQUE 5:')
print(f"- Mejor media en root_match_rate: {best_match_system}")
print(f"- Mejor media en cobertura: {best_cov_system}")
print(f"- Menor tiempo promedio: {fastest_system}")
print('- Esta lectura es descriptiva; la inferencia formal se evalúa en hipótesis globales y post-hoc.')

## 6. Comparar métricas lado a lado (3 sistemas)

Se construyen tablas de medias por métrica y deltas pareados entre pares de sistemas (`mse`, `match_count`, `sigmoid`).

In [ ]:
metrics_for_compare = ['root_match_rate', 'coverage_combined', 'time_seconds']

comparison_table = pd.DataFrame([
    {
        'metric': m,
        'mse_mean': float(df_all.loc[df_all['loss_mode'] == 'mse', m].mean()),
        'match_count_mean': float(df_all.loc[df_all['loss_mode'] == 'match_count', m].mean()),
        'sigmoid_mean': float(df_all.loc[df_all['loss_mode'] == 'sigmoid', m].mean()),
    }
    for m in metrics_for_compare
])

pairwise_descriptive = []
for m in metrics_for_compare:
    for a, b in PAIRS:
        a_vals = df_all.loc[df_all['loss_mode'] == a, m]
        b_vals = df_all.loc[df_all['loss_mode'] == b, m]
        mean_a = float(a_vals.mean())
        mean_b = float(b_vals.mean())
        delta = mean_b - mean_a
        pairwise_descriptive.append({
            'metric': m,
            'pair': f'{b} - {a}',
            'mean_a': mean_a,
            'mean_b': mean_b,
            'delta_mean': delta,
            'delta_rel_vs_a': delta / mean_a if np.isfinite(mean_a) and abs(mean_a) > 1e-12 else np.nan,
        })

pairwise_descriptive = pd.DataFrame(pairwise_descriptive)

wide_match = to_wide('root_match_rate')
wide_cov = to_wide('coverage_combined')
wide_time = to_wide('time_seconds')
wide_perfect = to_wide('perfect_case')
wide_no_sr = to_wide('no_sr_case')
wide_branch = to_wide('branch_count_correct')

print('Tabla comparativa de medias (3 sistemas):')
display(comparison_table)
print('Deltas descriptivos por pares:')
display(pairwise_descriptive)

In [ ]:
# Conclusiones del bloque 6 (comparación descriptiva 3 sistemas)
row_match = comparison_table.loc[comparison_table['metric'] == 'root_match_rate'].iloc[0]
row_cov = comparison_table.loc[comparison_table['metric'] == 'coverage_combined'].iloc[0]
row_time = comparison_table.loc[comparison_table['metric'] == 'time_seconds'].iloc[0]

best_match = pd.Series({
    'mse': row_match['mse_mean'],
    'match_count': row_match['match_count_mean'],
    'sigmoid': row_match['sigmoid_mean'],
}).idxmax()

best_cov = pd.Series({
    'mse': row_cov['mse_mean'],
    'match_count': row_cov['match_count_mean'],
    'sigmoid': row_cov['sigmoid_mean'],
}).idxmax()

fastest = pd.Series({
    'mse': row_time['mse_mean'],
    'match_count': row_time['match_count_mean'],
    'sigmoid': row_time['sigmoid_mean'],
}).idxmin()

print('CONCLUSION BLOQUE 6:')
print(f"- Mejor media descriptiva en root_match_rate: {best_match}")
print(f"- Mejor media descriptiva en cobertura: {best_cov}")
print(f"- Menor tiempo promedio: {fastest}")
print('- La significancia estadística de estas diferencias se evalúa en el bloque siguiente.')

## 7. Probar hipótesis continuas (Friedman + Wilcoxon post-hoc)

Se evalúan H1-H3 con:
- Test global de Friedman (3 sistemas pareados).
- Post-hoc Wilcoxon por pares con corrección de Holm.

In [ ]:
continuous_specs = [
    ('H1', 'root_match_rate'),
    ('H2', 'coverage_combined'),
    ('H3', 'time_seconds'),
]

continuous_global_rows = []
continuous_posthoc_rows = []

for hyp_id, metric in continuous_specs:
    wide = to_wide(metric)
    n = len(wide)

    if n >= 3:
        stat, pval = friedmanchisquare(wide['mse'], wide['match_count'], wide['sigmoid'])
        stat = float(stat)
        pval = float(pval)
    else:
        stat, pval = np.nan, np.nan

    continuous_global_rows.append({
        'hypothesis_id': hyp_id,
        'metric': metric,
        'n_pairs': int(n),
        'friedman_stat': stat,
        'friedman_pvalue': pval,
    })

    for a, b in PAIRS:
        if n >= 3:
            diff = wide[b] - wide[a]
            if np.allclose(diff.values, 0.0):
                w_stat, p_pair = 0.0, 1.0
            else:
                w_stat, p_pair = wilcoxon(diff.values, alternative='two-sided', zero_method='wilcox')
                w_stat = float(w_stat)
                p_pair = float(p_pair)
        else:
            w_stat, p_pair = np.nan, np.nan

        continuous_posthoc_rows.append({
            'hypothesis_id': hyp_id,
            'metric': metric,
            'pair': f'{b} vs {a}',
            'n_pairs': int(n),
            'wilcoxon_stat': w_stat,
            'pvalue_raw': p_pair,
            'mean_delta': float((wide[b] - wide[a]).mean()) if n > 0 else np.nan,
            'median_delta': float((wide[b] - wide[a]).median()) if n > 0 else np.nan,
        })

continuous_global_tests = pd.DataFrame(continuous_global_rows)
continuous_global_tests['reject_h0_alpha_0_05'] = continuous_global_tests['friedman_pvalue'] < ALPHA

continuous_posthoc = pd.DataFrame(continuous_posthoc_rows)
continuous_posthoc['pvalue_holm'] = np.nan
for metric, g in continuous_posthoc.groupby('metric'):
    idx = g.index
    corrected = holm_correction(g['pvalue_raw'].values)
    continuous_posthoc.loc[idx, 'pvalue_holm'] = corrected
continuous_posthoc['reject_h0_alpha_0_05'] = continuous_posthoc['pvalue_holm'] < ALPHA

print('Resultados globales (Friedman):')
display(continuous_global_tests)
print('Post-hoc pareado (Wilcoxon + Holm):')
display(continuous_posthoc)

In [ ]:
# Conclusiones del bloque 7 (hipótesis continuas H1-H3)
continuous_decision = continuous_global_tests[['hypothesis_id', 'metric', 'friedman_pvalue', 'reject_h0_alpha_0_05']].copy()
continuous_decision['interpretacion'] = np.where(
    continuous_decision['reject_h0_alpha_0_05'],
    'Se rechaza H0 global: hay diferencias entre al menos dos sistemas.',
    'No se rechaza H0 global: no hay evidencia suficiente de diferencias.'
)
display(continuous_decision)

for _, row in continuous_decision.iterrows():
    print(f"{row['hypothesis_id']} ({row['metric']}): {row['interpretacion']}")

print('Revisar la tabla post-hoc para identificar qué pares explican cada diferencia global.')

## 8. Probar hipótesis binarias (Cochran Q + McNemar post-hoc)

Se evalúan H4-H6 con:
- Test global de Cochran Q (3 sistemas pareados, variable binaria).
- Post-hoc McNemar exacto por pares con corrección de Holm.

In [ ]:
binary_specs = [
    ('H4', 'perfect_case', wide_perfect),
    ('H5', 'no_sr_case', wide_no_sr),
    ('H6', 'branch_count_correct', wide_branch),
]

binary_global_rows = []
binary_posthoc_rows = []

for hyp_id, metric, wide in binary_specs:
    mat = wide[SYSTEM_ORDER].copy()
    n = len(mat)

    q_stat, p_q = cochran_q_test(mat.values)

    binary_global_rows.append({
        'hypothesis_id': hyp_id,
        'metric': metric,
        'n_pairs': int(n),
        'cochran_q_stat': q_stat,
        'cochran_q_pvalue': p_q,
        'rate_mse': float(mat['mse'].mean()) if n > 0 else np.nan,
        'rate_match_count': float(mat['match_count'].mean()) if n > 0 else np.nan,
        'rate_sigmoid': float(mat['sigmoid'].mean()) if n > 0 else np.nan,
    })

    for a, b in PAIRS:
        p_pair, n01, n10, nd = pairwise_mcnemar_exact(mat[a].values, mat[b].values)
        binary_posthoc_rows.append({
            'hypothesis_id': hyp_id,
            'metric': metric,
            'pair': f'{b} vs {a}',
            'pvalue_raw': p_pair,
            'n01_a0_b1': n01,
            'n10_a1_b0': n10,
            'n_discordant': nd,
            'delta_rate': float(mat[b].mean() - mat[a].mean()) if n > 0 else np.nan,
        })

binary_global_tests = pd.DataFrame(binary_global_rows)
binary_global_tests['reject_h0_alpha_0_05'] = binary_global_tests['cochran_q_pvalue'] < ALPHA

binary_posthoc = pd.DataFrame(binary_posthoc_rows)
binary_posthoc['pvalue_holm'] = np.nan
for metric, g in binary_posthoc.groupby('metric'):
    idx = g.index
    corrected = holm_correction(g['pvalue_raw'].values)
    binary_posthoc.loc[idx, 'pvalue_holm'] = corrected
binary_posthoc['reject_h0_alpha_0_05'] = binary_posthoc['pvalue_holm'] < ALPHA

print('Resultados globales (Cochran Q):')
display(binary_global_tests)
print('Post-hoc (McNemar exacto + Holm):')
display(binary_posthoc)

In [ ]:
# Conclusiones del bloque 8 (hipótesis binarias H4-H6)
binary_decision = binary_global_tests[['hypothesis_id', 'metric', 'cochran_q_pvalue', 'reject_h0_alpha_0_05']].copy()
binary_decision['interpretacion'] = np.where(
    binary_decision['reject_h0_alpha_0_05'],
    'Se rechaza H0 global: hay diferencias entre sistemas para esta métrica binaria.',
    'No se rechaza H0 global: no hay evidencia suficiente de diferencias.'
)
display(binary_decision)

for _, row in binary_decision.iterrows():
    print(f"{row['hypothesis_id']} ({row['metric']}): {row['interpretacion']}")

# Consolidado de hipótesis H1-H6
hypo_global = pd.concat([
    continuous_global_tests[['hypothesis_id', 'metric', 'friedman_pvalue', 'reject_h0_alpha_0_05']]
        .rename(columns={'friedman_pvalue': 'pvalue_global'}),
    binary_global_tests[['hypothesis_id', 'metric', 'cochran_q_pvalue', 'reject_h0_alpha_0_05']]
        .rename(columns={'cochran_q_pvalue': 'pvalue_global'}),
], ignore_index=True)

HYPOTHESIS_RESULTS = HYPOTHESES.merge(
    hypo_global,
    left_on=['id', 'metric'],
    right_on=['hypothesis_id', 'metric'],
    how='left'
).drop(columns=['hypothesis_id'])

display(HYPOTHESIS_RESULTS)

## 9. Visualizar resultados y distribuciones

Se muestran distribuciones por sistema y deltas por pares para interpretar magnitud y dirección de efectos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df_all, x='loss_mode', y='root_match_rate', order=SYSTEM_ORDER, ax=axes[0])
axes[0].set_title('Root Match Rate')

sns.boxplot(data=df_all, x='loss_mode', y='coverage_combined', order=SYSTEM_ORDER, ax=axes[1])
axes[1].set_title('Coverage Combined')

sns.boxplot(data=df_all, x='loss_mode', y='time_seconds', order=SYSTEM_ORDER, ax=axes[2])
axes[2].set_title('Time (seconds)')

plt.tight_layout()
plt.show()

delta_rows = []
for metric, wide in [
    ('root_match_rate', wide_match),
    ('coverage_combined', wide_cov),
    ('time_seconds', wide_time),
]:
    for a, b in PAIRS:
        vals = (wide[b] - wide[a]).dropna().values
        for v in vals:
            delta_rows.append({'metric': metric, 'pair': f'{b}-{a}', 'delta': float(v)})

delta_df = pd.DataFrame(delta_rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['root_match_rate', 'coverage_combined', 'time_seconds']):
    subset = delta_df[delta_df['metric'] == metric]
    sns.histplot(data=subset, x='delta', hue='pair', kde=True, ax=ax)
    ax.axvline(0, color='black', linestyle='--')
    ax.set_title(f'Deltas pareados: {metric}')

plt.tight_layout()
plt.show()

## 10. Analizar errores por casos

Se identifica por caso qué sistema gana en calidad, y se inspeccionan casos problemáticos (`status != success` o `root_match_rate < 1`).

In [ ]:
quality_wide = to_wide('root_match_rate', dropna=False)
status_wide = (
    df_all
    .pivot_table(index=['name', 'repeat_id'], columns='loss_mode', values='status', aggfunc='first')
    .reindex(columns=SYSTEM_ORDER)
)

winner_rows = []
for idx, row in quality_wide.iterrows():
    valid = row.dropna()
    if valid.empty:
        continue
    max_val = float(valid.max())
    winners = [mode for mode, val in valid.items() if np.isclose(val, max_val)]
    winner_rows.append({
        'name': idx[0],
        'repeat_id': int(idx[1]),
        'best_root_match_rate': max_val,
        'winner_modes': ', '.join(winners),
        'n_winners': len(winners),
        'is_tie': len(winners) > 1,
    })

winner_df = pd.DataFrame(winner_rows)

problem_rows = []
for idx in quality_wide.index:
    case_name, rep_id = idx
    for mode in SYSTEM_ORDER:
        status = status_wide.loc[idx, mode] if idx in status_wide.index else np.nan
        rmr = quality_wide.loc[idx, mode]
        is_problem = (status != 'success') or (not np.isfinite(rmr)) or (float(rmr) < 1.0)
        if is_problem:
            problem_rows.append({
                'name': case_name,
                'repeat_id': int(rep_id),
                'loss_mode': mode,
                'status': status,
                'root_match_rate': float(rmr) if np.isfinite(rmr) else np.nan,
            })

problem_cases = (
    pd.DataFrame(problem_rows).sort_values(['name', 'repeat_id', 'loss_mode'])
    if problem_rows else
    pd.DataFrame(columns=['name', 'repeat_id', 'loss_mode', 'status', 'root_match_rate'])
)

print('Ganadores por caso/repetición (según root_match_rate):')
display(winner_df.sort_values(['is_tie', 'best_root_match_rate'], ascending=[True, False]).head(30))

print('Casos problemáticos (status != success o root_match_rate < 1):')
print('Total registros problemáticos:', len(problem_cases))
display(problem_cases.head(50))

In [ ]:
# Conclusiones del bloque 10 (errores y casos difíciles, 3 sistemas)
error_summary = (
    df_all.assign(is_problem=lambda d: (d['status'] != 'success') | (d['root_match_rate'] < 1.0))
          .groupby('loss_mode', as_index=False)
          .agg(
              n_total=('name', 'count'),
              n_problem=('is_problem', 'sum'),
              success_rate=('status', lambda s: float((s == 'success').mean())),
              mean_root_match_rate=('root_match_rate', 'mean'),
              mean_time_seconds=('time_seconds', 'mean'),
          )
)
error_summary['problem_rate'] = error_summary['n_problem'] / error_summary['n_total'].replace(0, np.nan)
display(error_summary)

clear_win_counts = (
    winner_df[winner_df['n_winners'] == 1]['winner_modes']
    .value_counts()
    .rename_axis('loss_mode')
    .reset_index(name='n_clear_wins')
)

tie_rate = float(winner_df['is_tie'].mean()) if not winner_df.empty else np.nan
print('Conteo de victorias claras por sistema (sin empates):')
display(clear_win_counts)
print(f'Tasa de empate en ganadores por caso/repetición: {tie_rate:.3f}' if np.isfinite(tie_rate) else 'Tasa de empate: NA')

if not clear_win_counts.empty:
    top_mode = clear_win_counts.iloc[0]['loss_mode']
    print(f'CONCLUSION BLOQUE 10: {top_mode} concentra más victorias claras en calidad por caso.')
else:
    print('CONCLUSION BLOQUE 10: no hay victorias claras suficientes para afirmar dominancia por caso.')

print(f'Registros problemáticos detectados: {len(problem_cases)}')

## 11. Generar tabla final y exportar reporte

Exporta datasets finales, tabla comparativa, estadística y figuras para revisión técnica.

In [ ]:
# Tabla final consolidada de hipótesis
final_table = HYPOTHESIS_RESULTS.copy()

# Unificar reportes globales y post-hoc
global_tests_report = pd.concat([
    continuous_global_tests.assign(test='friedman').rename(columns={'friedman_stat': 'stat', 'friedman_pvalue': 'pvalue_global'}),
    binary_global_tests.assign(test='cochran_q').rename(columns={'cochran_q_stat': 'stat', 'cochran_q_pvalue': 'pvalue_global'}),
], ignore_index=True, sort=False)

posthoc_report = pd.concat([
    continuous_posthoc.assign(test='wilcoxon'),
    binary_posthoc.assign(test='mcnemar_exact'),
], ignore_index=True, sort=False)

# Exportar
EXPORT_DIR = RESULTS_ROOT / f"analysis_report_3systems_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

df_mse.to_csv(EXPORT_DIR / 'system_mse.csv', index=False)
df_match.to_csv(EXPORT_DIR / 'system_match_count.csv', index=False)
df_sigmoid.to_csv(EXPORT_DIR / 'system_sigmoid.csv', index=False)
df_all.to_csv(EXPORT_DIR / 'all_runs.csv', index=False)

comparison_table.to_csv(EXPORT_DIR / 'comparison_table.csv', index=False)
pairwise_descriptive.to_csv(EXPORT_DIR / 'pairwise_descriptive.csv', index=False)

continuous_global_tests.to_csv(EXPORT_DIR / 'continuous_global_tests.csv', index=False)
continuous_posthoc.to_csv(EXPORT_DIR / 'continuous_posthoc.csv', index=False)
binary_global_tests.to_csv(EXPORT_DIR / 'binary_global_tests.csv', index=False)
binary_posthoc.to_csv(EXPORT_DIR / 'binary_posthoc.csv', index=False)

global_tests_report.to_csv(EXPORT_DIR / 'global_tests_report.csv', index=False)
posthoc_report.to_csv(EXPORT_DIR / 'posthoc_report.csv', index=False)

winner_df.to_csv(EXPORT_DIR / 'winner_by_case.csv', index=False)
problem_cases.to_csv(EXPORT_DIR / 'problem_cases.csv', index=False)
error_summary.to_csv(EXPORT_DIR / 'error_summary.csv', index=False)

final_table.to_csv(EXPORT_DIR / 'hypothesis_results.csv', index=False)

# Exportar todas las figuras abiertas
for idx, fig_num in enumerate(plt.get_fignums(), start=1):
    fig = plt.figure(fig_num)
    fig.savefig(EXPORT_DIR / f'figure_{idx}.png', dpi=150, bbox_inches='tight')

meta = {
    'generated_at': datetime.now().isoformat(),
    'project_root': str(PROJECT_ROOT),
    'results_root': str(RESULTS_ROOT),
    'export_dir': str(EXPORT_DIR),
    'source_folders': {mode: str(info['folder']) for mode, info in SYSTEM_MAP.items()},
    'systems': SYSTEM_ORDER,
    'alpha': ALPHA,
    'n_common_pairs': int(len(common_keys)),
    'n_total_rows': int(len(df_all)),
    'uses_precomputed_results_only': True,
}
with open(EXPORT_DIR / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('Reporte exportado en:', EXPORT_DIR)
display(final_table)

In [ ]:
# Conclusión global automatizada del experimento de 3 sistemas

def pick_winner_from_comparison(metric: str, lower_is_better: bool = False):
    row = comparison_table.loc[comparison_table['metric'] == metric]
    if row.empty:
        return None, {}
    r = row.iloc[0]
    values = {
        'mse': float(r['mse_mean']),
        'match_count': float(r['match_count_mean']),
        'sigmoid': float(r['sigmoid_mean']),
    }
    series = pd.Series(values, dtype=float).dropna()
    if series.empty:
        return None, values
    winner = series.idxmin() if lower_is_better else series.idxmax()
    return winner, values


def hypo_result(hyp_id: str):
    row = HYPOTHESIS_RESULTS.loc[HYPOTHESIS_RESULTS['id'] == hyp_id]
    if row.empty:
        return np.nan, False
    r = row.iloc[0]
    return float(r['pvalue_global']), bool(r['reject_h0_alpha_0_05'])


winner_match, vals_match = pick_winner_from_comparison('root_match_rate', lower_is_better=False)
winner_cov, vals_cov = pick_winner_from_comparison('coverage_combined', lower_is_better=False)
winner_time, vals_time = pick_winner_from_comparison('time_seconds', lower_is_better=True)

p_h1, sig_h1 = hypo_result('H1')
p_h2, sig_h2 = hypo_result('H2')
p_h3, sig_h3 = hypo_result('H3')
p_h4, sig_h4 = hypo_result('H4')
p_h5, sig_h5 = hypo_result('H5')
p_h6, sig_h6 = hypo_result('H6')

binary_rates = binary_global_tests.set_index('metric')[['rate_mse', 'rate_match_count', 'rate_sigmoid']] if not binary_global_tests.empty else pd.DataFrame()

def pick_binary(metric: str, lower_is_better: bool = False):
    if metric not in binary_rates.index:
        return None, {}
    row = binary_rates.loc[metric]
    values = {'mse': float(row['rate_mse']), 'match_count': float(row['rate_match_count']), 'sigmoid': float(row['rate_sigmoid'])}
    s = pd.Series(values, dtype=float).dropna()
    if s.empty:
        return None, values
    w = s.idxmin() if lower_is_better else s.idxmax()
    return w, values

winner_perfect, vals_perfect = pick_binary('perfect_case', lower_is_better=False)
winner_no_sr, vals_no_sr = pick_binary('no_sr_case', lower_is_better=True)
winner_branch, vals_branch = pick_binary('branch_count_correct', lower_is_better=False)

final_conclusion = {
    'winner_root_match_rate': winner_match,
    'winner_coverage': winner_cov,
    'winner_time_fastest': winner_time,
    'winner_perfect_case_rate': winner_perfect,
    'winner_lowest_no_sr_rate': winner_no_sr,
    'winner_branch_count_correct_rate': winner_branch,
    'pvalue_H1_root_match_rate': p_h1,
    'pvalue_H2_coverage': p_h2,
    'pvalue_H3_time': p_h3,
    'pvalue_H4_perfect_case': p_h4,
    'pvalue_H5_no_sr_case': p_h5,
    'pvalue_H6_branch_count_correct': p_h6,
    'reject_H1': sig_h1,
    'reject_H2': sig_h2,
    'reject_H3': sig_h3,
    'reject_H4': sig_h4,
    'reject_H5': sig_h5,
    'reject_H6': sig_h6,
}

final_conclusion_df = pd.DataFrame([final_conclusion])
display(final_conclusion_df)

print('RESUMEN EJECUTIVO AUTOMATIZADO (3 SISTEMAS)')
print(f"- Mejor desempeño descriptivo en root_match_rate: {winner_match} | valores={vals_match}")
print(f"- Mejor desempeño descriptivo en cobertura: {winner_cov} | valores={vals_cov}")
print(f"- Sistema más rápido (menor tiempo): {winner_time} | valores={vals_time}")
print(f"- Mayor tasa de casos perfectos: {winner_perfect} | valores={vals_perfect}")
print(f"- Menor tasa de no_sr_results: {winner_no_sr} | valores={vals_no_sr}")
print(f"- Mayor tasa de branch_count_correct: {winner_branch} | valores={vals_branch}")

print('Decisiones globales de hipótesis (alpha=0.05):')
for hyp_id, pval, sig in [
    ('H1', p_h1, sig_h1),
    ('H2', p_h2, sig_h2),
    ('H3', p_h3, sig_h3),
    ('H4', p_h4, sig_h4),
    ('H5', p_h5, sig_h5),
    ('H6', p_h6, sig_h6),
]:
    decision = 'rechaza H0' if sig else 'no rechaza H0'
    print(f"  - {hyp_id}: p={pval:.6g} -> {decision}")

export_dir_value = str(EXPORT_DIR) if 'EXPORT_DIR' in globals() else '(aún no exportado)'
print(f"- Artefactos exportados en: {export_dir_value}")